In [ ]:
import sys, warnings
sys.path.insert(0, 'src')
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import trimesh
import matplotlib.pyplot as plt

from MNM import solve_modified_newton_case

plt.rcParams.update({
    'figure.dpi':      150,
    'savefig.dpi':     200,
    'font.family':     'DejaVu Sans',
    'font.size':       11,
    'axes.grid':       True,
    'grid.alpha':      0.3,
    'lines.linewidth': 2.0,
    'lines.markersize': 6,
})

CAPSULA_DIR = Path('data/Capsula')
SAVE_DIR    = Path('results/Plots/AnalisisMallado')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MALLAS = [
    CAPSULA_DIR / 'ARD_coarsest.stl',
    CAPSULA_DIR / 'ARD_coarse.stl',
    CAPSULA_DIR / 'ARD_chillfine.stl',
    CAPSULA_DIR / 'ARD_fine.stl',
    CAPSULA_DIR / 'ARD_ultrafine.stl',
]

MACH_LIST   = [2, 3, 4, 6, 8, 10, 12, 15, 20]
ALPHA_FIXED = 10.0
GAMMA       = 1.4

In [ ]:
def wind_axes(alpha_deg):
    a  = np.deg2rad(alpha_deg)
    eD = np.array([0., -np.cos(a), -np.sin(a)]); eD /= np.linalg.norm(eD)
    eM = np.array([1., 0., 0.])
    eL = np.cross(eM, eD); eL /= np.linalg.norm(eL)
    return eD, eL, eM

def load_geom(path):
    m     = trimesh.load(str(path), force='mesh')
    verts = np.asarray(m.vertices, dtype=float)
    faces = np.asarray(m.faces,    dtype=int)
    v0, v1, v2 = verts[faces[:,0]], verts[faces[:,1]], verts[faces[:,2]]
    cross   = np.cross(v1-v0, v2-v0)
    areas   = np.linalg.norm(cross, axis=1) / 2.0
    normals = cross / (2*areas[:,None] + 1e-30)
    centers = (v0+v1+v2) / 3.0
    ext     = verts.max(0) - verts.min(0)
    return dict(
        centers=centers, areas=areas, normals=normals,
        S_ref=float(ext[0]*ext[2]), L_ref=float(ext[1]),
        r_ref=np.average(centers, axis=0, weights=areas),
        n_tri=len(faces),
    )

meshes = {}
for p in MALLAS:
    if not p.exists(): print(f'[SKIP] {p.name}'); continue
    d = load_geom(p)
    meshes[p.stem] = d
    print(f'  {p.stem:25s}  {d["n_tri"]:>8,} triÃ¡ngulos')

In [ ]:
eD, eL, eM = wind_axes(ALPHA_FIXED)

# res[nombre_malla] = {Mach: [], CD: [], CL: [], CM: [], cp_max: []}
res = {}
for name, d in meshes.items():
    Machs = []; CDs = []; CLs = []; CMs = []; CPs = []
    for M in MACH_LIST:
        r  = solve_modified_newton_case(
            centers=d['centers'], areas=d['areas'], normals=d['normals'],
            alpha_deg=ALPHA_FIXED, Mach=float(M),
            S_ref=d['S_ref'], L_ref=d['L_ref'], r_ref=d['r_ref'],
            eD=eD, eL=eL, eM=eM, gamma=GAMMA,
        )
        CF  = r['CF_total']; CMv = r['CM_total']
        Machs.append(M)
        CDs.append(float(np.dot(CF,  eD)))
        CLs.append(float(np.dot(CF,  eL)))
        CMs.append(float(np.dot(CMv, eM)))
        CPs.append(float(r['cp_max']))
    res[name] = dict(Mach=Machs, CD=CDs, CL=CLs, CM=CMs, cp_max=CPs)
    print(f'  {name} â€” OK')

In [ ]:
COLORS  = plt.cm.plasma(np.linspace(0.1, 0.85, len(meshes)))
MARKERS = ['o', 's', '^', 'D', 'v']

for coef, ylabel, fname in [
    ('CD',     'CD',     'MNM_CD_vs_Mach.png'),
    ('CL',     'CL',     'MNM_CL_vs_Mach.png'),
    ('CM',     'CM',     'MNM_CM_vs_Mach.png'),
    ('cp_max', 'Cp,max', 'MNM_Cpmax_vs_Mach.png'),
]:
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, (name, d) in enumerate(res.items()):
        lbl = f'ARD_{meshes[name]["n_tri"]:,}'
        ax.plot(d['Mach'], d[coef], marker=MARKERS[i], color=COLORS[i], label=lbl)
    ax.set_xlabel('M∞', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f'{ylabel} vs M∞  —  MNM  —  α = {ALPHA_FIXED}°', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(SAVE_DIR / fname)
    print(f'  → {SAVE_DIR / fname}')
    plt.show()